# 🛡️ Social Engineering Tactic Detector — Training Notebook

**What this notebook does:**
1. Checks GPU availability
2. Installs all dependencies
3. Downloads 7 real public datasets and maps them to our tactic taxonomy
4. Fine-tunes DistilBERT for multi-label tactic classification
5. Evaluates with per-class F1 scores
6. Downloads the trained model checkpoint as a `.zip`

**Tactic Taxonomy (Cialdini-based):**
| Tactic | Description |
|--------|-------------|
| `urgency` | Artificial time pressure — "act now", "24 hours" |
| `authority` | Impersonating banks, police, government |
| `isolation` | Discouraging victim from consulting others |
| `reciprocity` | Leveraging past favors to extract compliance |
| `emotional` | Exploiting fear, guilt, loneliness, romance |
| `benign` | Normal, non-manipulative conversation |

**Estimated time:** ~25–35 min on free T4 GPU

> ⚠️ **Before starting:** Go to `Runtime → Change runtime type → T4 GPU`

## ✅ Step 0 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('⚠️  No GPU found! Go to: Runtime → Change runtime type → T4 GPU')
    print('    Training will still work on CPU but will take ~2 hours.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 📦 Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install transformers==4.41.2 datasets==2.19.1 scikit-learn==1.5.0 shap==0.45.1
!pip install convokit  # for Persuasion for Good dataset
print('✅ All packages installed')

## 📥 Step 2 — Download & Preprocess Real Datasets

We pull from 7 sources and map their labels to our 6-class tactic taxonomy.

In [ ]:
import json, re, random, pathlib
from collections import Counter
from itertools import groupby
from datasets import load_dataset

TACTIC_LABELS = ['urgency', 'authority', 'isolation', 'reciprocity', 'emotional', 'benign']
LABEL2ID = {l: i for i, l in enumerate(TACTIC_LABELS)}
pathlib.Path('data').mkdir(exist_ok=True)

URGENCY_PATTERNS = [
    r'act now', r'immediately', r'within \d+ hours?', r'expires?',
    r'urgent', r'last chance', r'limited time', r'asap', r'deadline',
    r'suspended', r'blocked', r'right away'
]

def has_urgency(text):
    t = text.lower()
    return any(re.search(p, t) for p in URGENCY_PATTERNS)

ALL_RECORDS = []
print('✅ Imports ready')

In [ ]:
# ─── Dataset 1: MentalManip ───────────────────────────────────────────────────
print('Loading MentalManip…')

MENTALMANIP_MAP = {
    'persuasion': ['emotional'],
    'intimidation': ['authority', 'emotional'],
    'seduction': ['emotional', 'reciprocity'],
    'gaslighting': ['isolation', 'emotional'],
    'bribery': ['reciprocity'],
    'pretense': ['authority'],
    'emotional_blackmail': ['emotional', 'isolation'],
    'diversion': ['authority'],
    'shaming': ['emotional', 'isolation'],
}

try:
    mm_ds = load_dataset('audreyeleven/MentalManip', split='train', trust_remote_code=True)
    n = 0
    for i, row in enumerate(mm_ds):
        # Try multiple column names
        text = (row.get('sentence') or row.get('text') or row.get('utterance') or '').strip()
        if not text:
            continue
        manip_type = str(row.get('manipulation_type') or row.get('label') or '').lower()
        label_val = row.get('label', 1)
        if label_val == 0 or manip_type in ('none', '0', ''):
            tactics = ['benign']
        else:
            tactics = MENTALMANIP_MAP.get(manip_type, ['emotional'])
        ALL_RECORDS.append({
            'conversation_id': f'mm_{i:05d}', 'turn_id': 0,
            'speaker': 'Speaker', 'text': text,
            'tactics': tactics, 'source': 'mentalmanip'
        })
        n += 1
    print(f'  ✓ MentalManip: {n:,} examples')
except Exception as e:
    print(f'  ⚠ MentalManip failed: {e}')

In [ ]:
# ─── Dataset 2: Manipulative Language Detection ───────────────────────────────
print('Loading Manipulative Language Detection…')

PAULADRO_MAP = {
    'feigning_innocence': ['isolation'],
    'rationalization': ['authority'],
    'playing_victim': ['emotional', 'reciprocity'],
    'diversion': ['authority'],
    'coercion': ['urgency', 'emotional'],
    'normal': ['benign'],
    'non_manipulative': ['benign'],
    '0': ['benign'],
    '1': ['emotional'],
}

try:
    ml_ds = load_dataset('pauladroghoff/manipulative-language-detection', split='train', trust_remote_code=True)
    n = 0
    for i, row in enumerate(ml_ds):
        text = (row.get('text') or row.get('sentence') or '').strip()
        if not text:
            continue
        raw = str(row.get('label') or row.get('manipulation_type') or '').lower().strip()
        tactics = PAULADRO_MAP.get(raw, ['emotional'] if raw not in ('0', 'normal') else ['benign'])
        ALL_RECORDS.append({
            'conversation_id': f'ml_{i:05d}', 'turn_id': 0,
            'speaker': 'Speaker', 'text': text,
            'tactics': tactics, 'source': 'manipulative_language'
        })
        n += 1
    print(f'  ✓ Manipulative Language: {n:,} examples')
except Exception as e:
    print(f'  ⚠ Manipulative Language failed: {e}')

In [ ]:
# ─── Dataset 3: Phishing Dataset (emails + SMS) ───────────────────────────────
print('Loading Phishing Dataset…')

for subset, key in [('emails', 'ph'), ('sms', 'ph_sms')]:
    try:
        ph_ds = load_dataset('ealvaradob/phishing-dataset', subset, split='train', trust_remote_code=True)
        n = 0
        for i, row in enumerate(ph_ds):
            text = (row.get('text') or row.get('email') or row.get('sms') or '').strip()[:600]
            if not text or len(text) < 20:
                continue
            label = int(row.get('label', 0))
            if label == 1:
                tactics = ['authority']
                if has_urgency(text):
                    tactics.append('urgency')
            else:
                tactics = ['benign']
            ALL_RECORDS.append({
                'conversation_id': f'{key}_{i:05d}', 'turn_id': 0,
                'speaker': 'Sender', 'text': text,
                'tactics': tactics, 'source': f'phishing_{subset}'
            })
            n += 1
        print(f'  ✓ Phishing ({subset}): {n:,} examples')
    except Exception as e:
        print(f'  ⚠ Phishing ({subset}) failed: {e}')

In [ ]:
# ─── Dataset 4: Social Engineering Convo ─────────────────────────────────────
print('Loading Social Engineering Convo…')

SE_MAP = {
    'scam': ['urgency', 'authority'],
    'likely a scam': ['urgency'],
    'not a scam': ['benign'],
    '1': ['urgency', 'authority'],
    '0': ['benign'],
}

try:
    se_ds = load_dataset('Ngadou/social-engineering-convo', split='train', trust_remote_code=True)
    n = 0
    for i, row in enumerate(se_ds):
        text = (row.get('text') or row.get('conversation') or row.get('message') or '').strip()
        if not text:
            continue
        raw = str(row.get('label') or row.get('category') or '').lower().strip()
        tactics = SE_MAP.get(raw, ['urgency'])
        ALL_RECORDS.append({
            'conversation_id': f'se_{i:05d}', 'turn_id': 0,
            'speaker': 'Scammer', 'text': text,
            'tactics': tactics, 'source': 'social_engineering_convo'
        })
        n += 1
    print(f'  ✓ Social Engineering Convo: {n:,} examples')
except Exception as e:
    print(f'  ⚠ Social Engineering Convo failed: {e}')

In [ ]:
# ─── Dataset 5: SMS Spam Collection ──────────────────────────────────────────
print('Loading SMS Spam…')

for ds_name in ['sms_spam', 'ucirvine/sms_spam']:
    try:
        sms_ds = load_dataset(ds_name, split='train', trust_remote_code=True)
        n = 0
        for i, row in enumerate(sms_ds):
            text = (row.get('sms') or row.get('text') or row.get('message') or '').strip()
            label = row.get('label', 0)
            if not text:
                continue
            if label == 1:
                tactics = ['urgency'] if has_urgency(text) else ['authority']
            else:
                tactics = ['benign']
            ALL_RECORDS.append({
                'conversation_id': f'sms_{i:05d}', 'turn_id': 0,
                'speaker': 'Sender', 'text': text,
                'tactics': tactics, 'source': 'sms_spam'
            })
            n += 1
        print(f'  ✓ SMS Spam ({ds_name}): {n:,} examples')
        break
    except Exception as e:
        print(f'  ⚠ {ds_name} failed: {e}')

In [ ]:
# ─── Dataset 6: DailyDialog (benign class) ────────────────────────────────────
print('Loading DailyDialog (benign examples)…')
MAX_BENIGN = 3000

try:
    dd_ds = load_dataset('daily_dialog', split='train', trust_remote_code=True)
    n = 0
    conv_id = 0
    for dialog in dd_ds:
        for turn_id, text in enumerate(dialog.get('dialog', [])):
            if n >= MAX_BENIGN:
                break
            if not text.strip():
                continue
            ALL_RECORDS.append({
                'conversation_id': f'dd_{conv_id:05d}',
                'turn_id': turn_id,
                'speaker': 'Speaker' if turn_id % 2 == 0 else 'Listener',
                'text': text.strip(),
                'tactics': ['benign'],
                'source': 'daily_dialog'
            })
            n += 1
        conv_id += 1
        if n >= MAX_BENIGN:
            break
    print(f'  ✓ DailyDialog: {n:,} benign turns')
except Exception as e:
    print(f'  ⚠ DailyDialog failed: {e}')

In [ ]:
# ─── Dataset 7: Persuasion for Good (ConvoKit) ────────────────────────────────
print('Loading Persuasion for Good…')

P4G_MAP = {
    'foot-in-the-door': ['reciprocity'],
    'emotion appeal': ['emotional'],
    'logical appeal': ['benign'],
    'credibility appeal': ['authority'],
    'self-modeling': ['reciprocity'],
    'personal story': ['emotional'],
    'evidence': ['benign'],
    'donation information': ['benign'],
    'ask donation': ['reciprocity'],
    'acknowledgement': ['benign'],
    'no strategy': ['benign'],
    'have you heard': ['benign'],
    'task related': ['benign'],
}

try:
    from convokit import Corpus, download
    corpus = Corpus(filename=download('persuasionforgood-corpus'))
    n = 0
    for conv in corpus.iter_conversations():
        for utt in conv.iter_utterances():
            text = (utt.text or '').strip()
            if not text:
                continue
            strategy = str(
                utt.meta.get('er_label_1') or
                utt.meta.get('ee_label_1') or 'no strategy'
            ).lower().strip()
            tactics = P4G_MAP.get(strategy, ['emotional'])
            turn_id = int(utt.id.split('_')[-1]) if '_' in utt.id else n
            ALL_RECORDS.append({
                'conversation_id': f'p4g_{conv.id}',
                'turn_id': turn_id,
                'speaker': utt.speaker.id,
                'text': text,
                'tactics': tactics,
                'source': 'persuasion_for_good'
            })
            n += 1
    print(f'  ✓ Persuasion for Good: {n:,} turns')
except Exception as e:
    print(f'  ⚠ Persuasion for Good failed: {e}')

In [ ]:
# ─── Dataset Summary ──────────────────────────────────────────────────────────
import random
random.shuffle(ALL_RECORDS)

tactic_counts = Counter(t for r in ALL_RECORDS for t in r['tactics'])
source_counts = Counter(r['source'] for r in ALL_RECORDS)

print(f'\n{'='*50}')
print(f'TOTAL RECORDS: {len(ALL_RECORDS):,}')
print(f'\nPer tactic:')
for t in TACTIC_LABELS:
    c = tactic_counts.get(t, 0)
    bar = '█' * (c // max(max(tactic_counts.values())//30, 1))
    print(f'  {t:<14} {c:>7,}  {bar}')

print(f'\nPer source:')
for s, c in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f'  {s:<35} {c:>7,}')

# Save to disk
with open('data/real_data.jsonl', 'w') as f:
    for r in ALL_RECORDS:
        f.write(json.dumps(r) + '\n')
print(f'\n✅ Saved to data/real_data.jsonl')

## 🤗 Step 3 — Build Dataset & DataLoaders

In [ ]:
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

MAX_LEN = 256
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─── Context map: conv_id → ordered list of turn texts ────────────────────────
def build_context_map(records):
    ctx = {}
    for r in records:
        ctx.setdefault(r['conversation_id'], []).append((r.get('turn_id', 0), r['text']))
    # Sort each conversation by turn_id
    return {cid: [t for _, t in sorted(turns)] for cid, turns in ctx.items()}

context_map = build_context_map(ALL_RECORDS)

class TacticDataset(Dataset):
    def __init__(self, records, tokenizer, context_map):
        self.records = records
        self.tokenizer = tokenizer
        self.context_map = context_map

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        turn_id = int(rec.get('turn_id', 0))
        prior = self.context_map.get(rec['conversation_id'], [])
        context = prior[max(0, turn_id - 2): turn_id]
        combined = ' [SEP] '.join(context + [rec['text']])

        enc = self.tokenizer(
            combined, max_length=MAX_LEN, truncation=True,
            padding='max_length', return_tensors='pt'
        )
        label_vec = torch.zeros(len(TACTIC_LABELS))
        for t in rec.get('tactics', ['benign']):
            if t in LABEL2ID:
                label_vec[LABEL2ID[t]] = 1.0

        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': label_vec
        }

print('✅ Dataset class defined')

In [ ]:
# ─── Train / Val / Test split ─────────────────────────────────────────────────
train_recs, test_recs = train_test_split(ALL_RECORDS, test_size=0.2, random_state=SEED)
val_recs, test_recs = train_test_split(test_recs, test_size=0.5, random_state=SEED)
print(f'Train: {len(train_recs):,} | Val: {len(val_recs):,} | Test: {len(test_recs):,}')

# ─── Tokenizer ────────────────────────────────────────────────────────────────
BASE_MODEL = 'distilbert-base-uncased'   # swap to 'roberta-base' for +3-5% F1
print(f'\nLoading tokenizer: {BASE_MODEL}…')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_ds = TacticDataset(train_recs, tokenizer, context_map)
val_ds   = TacticDataset(val_recs,   tokenizer, context_map)
test_ds  = TacticDataset(test_recs,  tokenizer, context_map)

# ─── Weighted sampler for class imbalance ─────────────────────────────────────
tactic_freq = Counter(t for r in train_recs for t in r.get('tactics', ['benign']))
sample_weights = [1.0 / min(tactic_freq.get(t, 1) for t in r.get('tactics', ['benign'])) for r in train_recs]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

BATCH_SIZE = 32  # safe for T4 16GB; reduce to 16 if you see OOM
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'✅ DataLoaders ready (batch_size={BATCH_SIZE})')

## 🚀 Step 4 — Fine-tune DistilBERT

In [ ]:
EPOCHS = 5
LR     = 2e-5
THRESHOLD = 0.5

print(f'Loading model: {BASE_MODEL}…')
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(TACTIC_LABELS),
    problem_type='multi_label_classification'
).to(device)

# Weighted BCE loss
total = len(train_recs)
pos_weights = torch.tensor([
    max((total - tactic_freq.get(l, 1)) / tactic_freq.get(l, 1), 1.0)
    for l in TACTIC_LABELS
], dtype=torch.float).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

print(f'✅ Model loaded ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)')
print(f'   Training for {EPOCHS} epochs, {len(train_loader)} steps/epoch')

In [ ]:
import pathlib
output_dir = pathlib.Path('checkpoints/best_model')
output_dir.mkdir(parents=True, exist_ok=True)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device)
            ).logits
            preds = (torch.sigmoid(logits) >= THRESHOLD).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].numpy())
    return f1_score(all_labels, all_preds, average='macro', zero_division=0), all_preds, all_labels

best_val_f1 = 0.0
history = {'train_loss': [], 'val_f1': []}

print(f'🚀 Starting training on {device}…\n')
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device)
        )
        loss = criterion(outputs.logits, batch['labels'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if (step + 1) % 100 == 0:
            print(f'  Ep {epoch} | Step {step+1}/{len(train_loader)} | loss={total_loss/(step+1):.4f}')

    avg_loss = total_loss / len(train_loader)
    val_f1, _, _ = evaluate(model, val_loader)
    history['train_loss'].append(avg_loss)
    history['val_f1'].append(val_f1)

    print(f'\n{'='*55}')
    print(f'Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  val_macro_F1={val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        print(f'  💾 Saved best model (F1={val_f1:.4f})')
    print(f'{'='*55}\n')

print(f'\n✅ Training complete! Best val macro-F1: {best_val_f1:.4f}')

## 📊 Step 5 — Evaluate on Test Set

In [ ]:
print('Loading best checkpoint for final evaluation…')
best_model = AutoModelForSequenceClassification.from_pretrained(
    output_dir, num_labels=len(TACTIC_LABELS)
).to(device)

_, preds, labels = evaluate(best_model, test_loader)

print('\n' + '='*55)
print('📊 TEST SET CLASSIFICATION REPORT')
print('='*55)
print(classification_report(labels, preds, target_names=TACTIC_LABELS, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0a0a0f')
for ax in [ax1, ax2]:
    ax.set_facecolor('#12121a')
    ax.tick_params(colors='#e8e8f0')
    ax.spines[:].set_color('#2a2a3a')
    ax.xaxis.label.set_color('#e8e8f0')
    ax.yaxis.label.set_color('#e8e8f0')
    ax.title.set_color('#e8e8f0')

epochs_range = range(1, EPOCHS + 1)
ax1.plot(epochs_range, history['train_loss'], 'o-', color='#f59e0b', linewidth=2, label='Train Loss')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.legend()

ax2.plot(epochs_range, history['val_f1'], 's-', color='#8b5cf6', linewidth=2, label='Val Macro F1')
ax2.axhline(y=best_val_f1, color='#10b981', linestyle='--', alpha=0.6, label=f'Best={best_val_f1:.3f}')
ax2.set_title('Validation Macro F1')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('✅ Saved training_curves.png')

## 🧪 Step 6 — Quick Inference Test

In [ ]:
best_model.eval()

TEST_TURNS = [
    'This is urgent — your account will be suspended in 24 hours if you do not act now.',
    'I am calling from Microsoft Security Division. We have detected a virus on your computer.',
    'Please do not tell anyone about this. Keep it just between us.',
    'After everything I have done for you, this is the least you can do for me.',
    'I am so scared and alone. You are the only one who understands me.',
    'Hey, do you want to grab lunch today? I was thinking maybe the Italian place.',
]

print('='*60)
print('QUICK INFERENCE TEST')
print('='*60)

TACTIC_ICONS = {
    'urgency': '⏰', 'authority': '🎖️', 'isolation': '🔒',
    'reciprocity': '🤝', 'emotional': '💔', 'benign': '✅'
}

for text in TEST_TURNS:
    enc = tokenizer(text, return_tensors='pt', max_length=MAX_LEN, truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = best_model(**enc).logits
    probs = torch.sigmoid(logits).squeeze().cpu().tolist()
    detected = [(TACTIC_LABELS[i], p) for i, p in enumerate(probs) if p >= THRESHOLD]
    if not detected:
        detected = [('benign', max(probs))]
    print(f'\nText: "{text[:70]}{'...' if len(text)>70 else ''}"')
    for tactic, conf in sorted(detected, key=lambda x: -x[1]):
        icon = TACTIC_ICONS.get(tactic, '?')
        bar = '█' * int(conf * 20)
        print(f'  {icon} {tactic:<14} {conf:.3f}  {bar}')

## 💾 Step 7 — Download the Trained Model

In [ ]:
import shutil
from google.colab import files

# Zip the checkpoint
zip_path = 'tactic_detector_model'
shutil.make_archive(zip_path, 'zip', 'checkpoints', 'best_model')
print(f'✅ Created {zip_path}.zip')

# Also save training curves
shutil.copy('training_curves.png', f'{zip_path}_curves.png')

# Download both
files.download(f'{zip_path}.zip')
files.download(f'{zip_path}_curves.png')
print('📥 Download started!')

## 🔌 Step 8 — Use the Model in Your FastAPI Backend

After downloading `tactic_detector_model.zip`:

1. **Unzip** into `backend/training/checkpoints/best_model/`

2. **Edit one line** in `backend/app/main.py`:
```python
# Change:
from app.mock_inference import analyze_transcript, compute_risk_score, get_dominant_tactic
# To:
from app.inference import analyze_transcript, compute_risk_score, get_dominant_tactic
```

3. **Set the model path** when starting the server:
```bash
MODEL_PATH=./training/checkpoints/best_model uvicorn app.main:app --reload
```

4. **Optional: push to HuggingFace Hub**
```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained('checkpoints/best_model')
model.push_to_hub('meetmehedi/tactic-detector')
tokenizer.push_to_hub('meetmehedi/tactic-detector')
```

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment and fill in your HF token

# HF_TOKEN = 'hf_your_token_here'
# REPO_NAME = 'meetmehedi/tactic-detector'

# from huggingface_hub import login
# login(token=HF_TOKEN)

# best_model.push_to_hub(REPO_NAME)
# tokenizer.push_to_hub(REPO_NAME)
# print(f'✅ Model pushed to https://huggingface.co/{REPO_NAME}')

print('HuggingFace push is commented out. Uncomment lines above to use.')